# Load & Test — xgb_v4_OTA2201_20260602_1457

Loads the saved XGBoost model and runs a rolling 1-week prediction
on the first week of August 2024.

Preprocessing is identical to `xgboost_v4_search.ipynb` (v3 pipeline on old data).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
# ── Load saved model ──────────────────────────────────────────────────────────

MODEL_PATH = "./xgb_v4_OTA2201_20260602_1457.ubj"

model = XGBRegressor()
model.load_model(MODEL_PATH)

print(f"Model loaded: {MODEL_PATH}")

In [ ]:
# ── Load data — identical to v4_search ───────────────────────────────────────

df = pd.read_csv('../../Data_Processing/preprocessed_data.csv')
df["datetime_utc12"] = pd.to_datetime(df["datetime_utc12"])
df = df.sort_values("datetime_utc12").reset_index(drop=True)

target_col = "el_price_dol_MWh_OTA2201"
print("Shape:", df.shape)

In [ ]:
# ── Preprocessing — identical to v4_search ────────────────────────────────────

df = df.ffill()

cols_to_shift_24h = [
    c for c in df.columns if "el_price" in c and c != target_col
] + [
    "Coal","Diesel","Ele","Gas","Geo","Hydro","Solar","Wind","Wood",
    "demand_GWh_CNI","demand_GWh_LNI","demand_GWh_LSI","demand_GWh_UNI","demand_GWh_USI",
    "avg_flow_MW","peak_flow_MW","Direction"
]
for col in cols_to_shift_24h:
    df[f"{col}_lag24"] = df[col].shift(24)
    df = df.drop(columns=[col])

df["hour"]       = df["datetime_utc12"].dt.hour
df["dayofweek"]  = df["datetime_utc12"].dt.dayofweek
df["month"]      = df["datetime_utc12"].dt.month
df["dayofyear"]  = df["datetime_utc12"].dt.dayofyear
df["is_weekend"] = df["dayofweek"].isin([5,6]).astype(int)
df["hour_sin"]   = np.sin(2*np.pi*df["hour"]/24)
df["hour_cos"]   = np.cos(2*np.pi*df["hour"]/24)
df["dow_sin"]    = np.sin(2*np.pi*df["dayofweek"]/7)
df["dow_cos"]    = np.cos(2*np.pi*df["dayofweek"]/7)
df["month_sin"]  = np.sin(2*np.pi*df["month"]/12)
df["month_cos"]  = np.cos(2*np.pi*df["month"]/12)

df["target_lag_24h"]   = df[target_col].shift(24)
df["target_lag_168h"]  = df[target_col].shift(168)
df["target_lag_8760h"] = df[target_col].shift(8760)

for window in [24, 168, 8760]:
    shifted = df[target_col].shift(1)
    df[f"rolling_mean_{window}h"] = shifted.rolling(window).mean()
    df[f"rolling_std_{window}h"]  = shifted.rolling(window).std()

df = df.dropna().reset_index(drop=True)
print(f"Rows after cleanup: {len(df)}")
print(f"Date range: {df['datetime_utc12'].iloc[0].date()} → {df['datetime_utc12'].iloc[-1].date()}")

In [ ]:
# ── Define feature columns ────────────────────────────────────────────────────

exclude_cols = ["datetime_utc12", target_col]
feature_cols = [c for c in df.columns if c not in exclude_cols]

print(f"Features: {len(feature_cols)}")

In [ ]:
# ── Predict every day of 2024 ─────────────────────────────────────────────────

results = []

dates_2024 = pd.date_range("2024-01-01", "2024-12-31", freq="D")

for date in dates_2024:
    mask = (
        (df["datetime_utc12"] >= date) &
        (df["datetime_utc12"] <  date + pd.Timedelta(hours=24))
    )
    day_df = df[mask]

    if len(day_df) < 24:
        continue

    X_day    = day_df[feature_cols].values
    y_actual = day_df[target_col].values
    y_naive  = day_df["target_lag_24h"].values
    y_pred   = model.predict(X_day)

    mae_model = mean_absolute_error(y_actual, y_pred)
    mae_naive = mean_absolute_error(y_actual, y_naive)
    std_price = np.std(y_actual)
    max_price = np.max(y_actual)

    results.append({
        "date":              date.date(),
        "mae_model":         mae_model,
        "mae_naive":         mae_naive,
        "std_actual":        std_price,
        "max_actual":        max_price,
        "beats_naive":       mae_model < mae_naive,
        "score_std_per_mae": std_price / mae_model if mae_model > 0 else np.nan,
    })

results_df = pd.DataFrame(results)
results_df["naive_advantage"] = results_df["mae_naive"] - results_df["mae_model"]

print(f"Days predicted      : {len(results_df)}")
print(f"Days beats naive    : {results_df['beats_naive'].sum()} / {len(results_df)}")
print(f"Days with max > 1000: {(results_df['max_actual'] > 1000).sum()}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# RANKING 1 — TOP 10 DAYS: Lowest Model MAE
# Most accurate days overall regardless of volatility
# ══════════════════════════════════════════════════════════════════════════════

pd.set_option("display.float_format", "{:.2f}".format)

top_mae = results_df.nsmallest(10, "mae_model")[
    ["date", "mae_model", "mae_naive", "std_actual", "beats_naive"]
].reset_index(drop=True)
top_mae.index += 1
print(top_mae.to_string())

# ── Plot top 3 ────────────────────────────────────────────────────────────────

def plot_day(ax, date, title):
    mask = (
        (df["datetime_utc12"] >= pd.Timestamp(date)) &
        (df["datetime_utc12"] <  pd.Timestamp(date) + pd.Timedelta(hours=24))
    )
    d        = df[mask]
    ts       = d["datetime_utc12"].values
    y_actual = d[target_col].values
    y_pred   = model.predict(d[feature_cols].values)
    y_naive  = d["target_lag_24h"].values
    mae_m    = mean_absolute_error(y_actual, y_pred)
    mae_n    = mean_absolute_error(y_actual, y_naive)

    ax.plot(ts, y_actual, label="Actual",  color="steelblue",  linewidth=1.5)
    ax.plot(ts, y_pred,   label="XGBoost", color="darkorange", linewidth=1.5, linestyle="--")
    ax.plot(ts, y_naive,  label="Naive",   color="gray",       linewidth=1.0, linestyle=":")
    ax.set_title(f"{title}\nMAE model: {mae_m:.1f}  naive: {mae_n:.1f}", fontsize=9)
    ax.set_ylabel("Price (NZD/MWh)")
    ax.set_ylim(bottom=0)
    ax.tick_params(axis="x", labelrotation=30, labelsize=7)
    ax.legend(fontsize=7)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle("Ranking 1 — Top 3 Most Accurate Days (Lowest MAE)", fontsize=12)
for i, ax in enumerate(axes):
    date = top_mae.iloc[i]["date"]
    plot_day(ax, date, str(date))
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# RANKING 2 — TOP 10 DAYS: Model Beats Naive by Largest Margin
# Days where the model added the most value over simply using yesterday
# (only days where model actually wins)
# ══════════════════════════════════════════════════════════════════════════════

beats = results_df[results_df["beats_naive"]].nlargest(10, "naive_advantage")[
    ["date", "mae_model", "mae_naive", "naive_advantage", "std_actual"]
].reset_index(drop=True)
beats.index += 1
print(beats.to_string())

# ── Plot top 3 ────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle("Ranking 2 — Top 3 Days: Model Beats Naive by Largest Margin", fontsize=12)
for i, ax in enumerate(axes):
    date = beats.iloc[i]["date"]
    plot_day(ax, date, str(date))
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# RANKING 3 — VOLATILE BUT ACCURATE: score = std / model_MAE
# Excludes days where any hour exceeded 1000 NZD/MWh — those days have
# artificially large std from a single extreme spike, not genuine volatility.
# ══════════════════════════════════════════════════════════════════════════════

SPIKE_EXCLUDE = 1000   # NZD/MWh — tweak if needed

r3 = results_df[results_df["max_actual"] <= SPIKE_EXCLUDE].copy()
excluded = len(results_df) - len(r3)
print(f"Days excluded (max price > {SPIKE_EXCLUDE}): {excluded}  |  Remaining: {len(r3)}\n")

top_score  = r3.nlargest(10,  "score_std_per_mae")
worst_score = r3.nsmallest(10, "score_std_per_mae")

cols_show = ["date", "score_std_per_mae", "std_actual", "max_actual", "mae_model", "mae_naive", "beats_naive"]

print("── TOP 10: Volatile days where model was most accurate ──")
ts_print = top_score[cols_show].reset_index(drop=True)
ts_print.index += 1
print(ts_print.to_string())

print("\n── WORST 10: Volatile days where model struggled most ──")
ws_print = worst_score[cols_show].reset_index(drop=True)
ws_print.index += 1
print(ws_print.to_string())

fig, axes = plt.subplots(1, 5, figsize=(25, 4))
fig.suptitle(f"Ranking 3 — Top 5: Volatile Days, Model Accurate (max price ≤ {SPIKE_EXCLUDE})", fontsize=11)
for i, ax in enumerate(axes):
    date  = top_score.iloc[i]["date"]
    score = top_score.iloc[i]["score_std_per_mae"]
    plot_day(ax, date, f"{date}\nscore: {score:.1f}")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 5, figsize=(25, 4))
fig.suptitle(f"Ranking 3 — Worst 5: Volatile Days, Model Struggled (max price ≤ {SPIKE_EXCLUDE})", fontsize=11)
for i, ax in enumerate(axes):
    date  = worst_score.iloc[i]["date"]
    score = worst_score.iloc[i]["score_std_per_mae"]
    plot_day(ax, date, f"{date}\nscore: {score:.1f}")
plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("FULL YEAR 2024 — Summary")
print("=" * 60)
print(f"  Days predicted    : {len(results_df)}")
print(f"  Avg model MAE     : {results_df['mae_model'].mean():.2f} NZD/MWh")
print(f"  Avg naive MAE     : {results_df['mae_naive'].mean():.2f} NZD/MWh")
print(f"  Avg MASE          : {(results_df['mae_model'] / results_df['mae_naive']).mean():.3f}")
print(f"  Days beats naive  : {results_df['beats_naive'].sum()} / {len(results_df)}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# WEEKLY ANALYSIS — Predict each week of 2024
# ══════════════════════════════════════════════════════════════════════════════

week_results = []

week_starts = pd.date_range("2024-01-01", "2024-12-25", freq="7D")

for week_start in week_starts:
    week_end = week_start + pd.Timedelta(days=7)
    mask = (
        (df["datetime_utc12"] >= week_start) &
        (df["datetime_utc12"] <  week_end)
    )
    week_df = df[mask]

    if len(week_df) < 168:
        continue

    X_week   = week_df[feature_cols].values
    y_actual = week_df[target_col].values
    y_naive  = week_df["target_lag_24h"].values
    y_pred   = model.predict(X_week)

    mae_model = mean_absolute_error(y_actual, y_pred)
    mae_naive = mean_absolute_error(y_actual, y_naive)
    std_price = np.std(y_actual)
    max_price = np.max(y_actual)

    week_results.append({
        "week_start":        week_start.date(),
        "week_end":          (week_end - pd.Timedelta(days=1)).date(),
        "mae_model":         mae_model,
        "mae_naive":         mae_naive,
        "std_actual":        std_price,
        "max_actual":        max_price,
        "beats_naive":       mae_model < mae_naive,
        "score_std_per_mae": std_price / mae_model if mae_model > 0 else np.nan,
    })

week_df_results = pd.DataFrame(week_results)
week_df_results["naive_advantage"] = week_df_results["mae_naive"] - week_df_results["mae_model"]

print(f"Weeks predicted      : {len(week_df_results)}")
print(f"Weeks beats naive    : {week_df_results['beats_naive'].sum()} / {len(week_df_results)}")
print(f"Weeks with max > 1000: {(week_df_results['max_actual'] > 1000).sum()}")

def plot_week(ax, week_start, title):
    week_end = pd.Timestamp(week_start) + pd.Timedelta(days=7)
    mask = (
        (df["datetime_utc12"] >= pd.Timestamp(week_start)) &
        (df["datetime_utc12"] <  week_end)
    )
    d        = df[mask]
    ts       = d["datetime_utc12"].values
    y_actual = d[target_col].values
    y_pred   = model.predict(d[feature_cols].values)
    y_naive  = d["target_lag_24h"].values
    mae_m    = mean_absolute_error(y_actual, y_pred)
    mae_n    = mean_absolute_error(y_actual, y_naive)

    ax.plot(ts, y_actual, label="Actual",  color="steelblue",  linewidth=1.0)
    ax.plot(ts, y_pred,   label="XGBoost", color="darkorange", linewidth=1.0, linestyle="--")
    ax.plot(ts, y_naive,  label="Naive",   color="gray",       linewidth=0.8, linestyle=":")
    ax.set_title(f"{title}\nMAE model: {mae_m:.1f}  naive: {mae_n:.1f}", fontsize=8)
    ax.set_ylabel("Price (NZD/MWh)")
    ax.set_ylim(bottom=0)
    ax.tick_params(axis="x", labelrotation=30, labelsize=6)
    ax.legend(fontsize=6)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# WEEKLY RANKING 1 — TOP 10 WEEKS: Lowest Model MAE
# ══════════════════════════════════════════════════════════════════════════════

top_mae_w = week_df_results.nsmallest(10, "mae_model")[
    ["week_start", "week_end", "mae_model", "mae_naive", "std_actual", "beats_naive"]
].reset_index(drop=True)
top_mae_w.index += 1
print(top_mae_w.to_string())

fig, axes = plt.subplots(1, 3, figsize=(20, 4))
fig.suptitle("Weekly Ranking 1 — Top 3 Most Accurate Weeks (Lowest MAE)", fontsize=12)
for i, ax in enumerate(axes):
    ws = top_mae_w.iloc[i]["week_start"]
    plot_week(ax, ws, str(ws))
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# WEEKLY RANKING 2 — TOP 10 WEEKS: Model Beats Naive by Largest Margin
# ══════════════════════════════════════════════════════════════════════════════

beats_w = week_df_results[week_df_results["beats_naive"]].nlargest(10, "naive_advantage")[
    ["week_start", "week_end", "mae_model", "mae_naive", "naive_advantage", "std_actual"]
].reset_index(drop=True)
beats_w.index += 1
print(beats_w.to_string())

fig, axes = plt.subplots(1, 3, figsize=(20, 4))
fig.suptitle("Weekly Ranking 2 — Top 3 Weeks: Model Beats Naive by Largest Margin", fontsize=12)
for i, ax in enumerate(axes):
    ws = beats_w.iloc[i]["week_start"]
    plot_week(ax, ws, str(ws))
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# WEEKLY RANKING 3 — VOLATILE BUT ACCURATE: score = std / model_MAE
# Excludes weeks where any hour exceeded 1000 NZD/MWh
# ══════════════════════════════════════════════════════════════════════════════

SPIKE_EXCLUDE = 1000   # NZD/MWh — same threshold as daily

r3w = week_df_results[week_df_results["max_actual"] <= SPIKE_EXCLUDE].copy()
excluded_w = len(week_df_results) - len(r3w)
print(f"Weeks excluded (max price > {SPIKE_EXCLUDE}): {excluded_w}  |  Remaining: {len(r3w)}\n")

top_score_w   = r3w.nlargest(10,  "score_std_per_mae")
worst_score_w = r3w.nsmallest(10, "score_std_per_mae")

cols_show_w = ["week_start", "week_end", "score_std_per_mae", "std_actual", "max_actual", "mae_model", "mae_naive", "beats_naive"]

print("── TOP 10 WEEKS: Volatile weeks where model was most accurate ──")
ts_w = top_score_w[cols_show_w].reset_index(drop=True)
ts_w.index += 1
print(ts_w.to_string())

print("\n── WORST 10 WEEKS: Volatile weeks where model struggled most ──")
ws_w = worst_score_w[cols_show_w].reset_index(drop=True)
ws_w.index += 1
print(ws_w.to_string())

fig, axes = plt.subplots(1, 5, figsize=(28, 4))
fig.suptitle(f"Weekly Ranking 3 — Top 5: Volatile Weeks, Model Accurate (max price ≤ {SPIKE_EXCLUDE})", fontsize=11)
for i, ax in enumerate(axes):
    ws    = top_score_w.iloc[i]["week_start"]
    score = top_score_w.iloc[i]["score_std_per_mae"]
    plot_week(ax, ws, f"{ws}\nscore: {score:.1f}")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 5, figsize=(28, 4))
fig.suptitle(f"Weekly Ranking 3 — Worst 5: Volatile Weeks, Model Struggled (max price ≤ {SPIKE_EXCLUDE})", fontsize=11)
for i, ax in enumerate(axes):
    ws    = worst_score_w.iloc[i]["week_start"]
    score = worst_score_w.iloc[i]["score_std_per_mae"]
    plot_week(ax, ws, f"{ws}\nscore: {score:.1f}")
plt.tight_layout()
plt.show()

print("\n" + "=" * 60)
print("FULL YEAR 2024 — Weekly Summary")
print("=" * 60)
print(f"  Weeks predicted    : {len(week_df_results)}")
print(f"  Avg model MAE      : {week_df_results['mae_model'].mean():.2f} NZD/MWh")
print(f"  Avg naive MAE      : {week_df_results['mae_naive'].mean():.2f} NZD/MWh")
print(f"  Avg MASE           : {(week_df_results['mae_model'] / week_df_results['mae_naive']).mean():.3f}")
print(f"  Weeks beats naive  : {week_df_results['beats_naive'].sum()} / {len(week_df_results)}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ALL 52 WEEKS OF 2024 — One plot per week
# ══════════════════════════════════════════════════════════════════════════════

for _, row in week_df_results.iterrows():
    ws    = row["week_start"]
    we    = row["week_end"]
    mae_m = row["mae_model"]
    mae_n = row["mae_naive"]
    mase  = mae_m / mae_n
    flag  = "✓" if row["beats_naive"] else "✗"

    fig, ax = plt.subplots(figsize=(16, 4))
    plot_week(ax, ws, f"{ws} → {we}")
    ax.set_title(
        f"{ws} → {we}  |  "
        f"MAE model: {mae_m:.1f}  naive: {mae_n:.1f}  "
        f"MASE: {mase:.3f}  {flag}  |  "
        f"std: {row['std_actual']:.1f}  max: {row['max_actual']:.0f}",
        fontsize=9
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# OPTIMAL CHARGING SCHEDULER
# ══════════════════════════════════════════════════════════════════════════════

# ── USER INPUTS ───────────────────────────────────────────────────────────────

WEEK_START       = "2024-08-05"
DAILY_ENERGY_MWH = 2.0
MACHINE_POWER_MW = 0.5

# ── DERIVED ───────────────────────────────────────────────────────────────────

HOURS_PER_DAY = int(np.ceil(DAILY_ENERGY_MWH / MACHINE_POWER_MW))
print(f"Daily energy required : {DAILY_ENERGY_MWH} MWh")
print(f"Machine power         : {MACHINE_POWER_MW} MW")
print(f"Consumption hours/day : {HOURS_PER_DAY}h  "
      f"({MACHINE_POWER_MW} MW × {HOURS_PER_DAY}h = "
      f"{MACHINE_POWER_MW * HOURS_PER_DAY:.1f} MWh/day)")

# ── FETCH WEEK DATA ───────────────────────────────────────────────────────────

week_start_ts = pd.Timestamp(WEEK_START)
week_end_ts   = week_start_ts + pd.Timedelta(days=7)

mask    = (df["datetime_utc12"] >= week_start_ts) & (df["datetime_utc12"] < week_end_ts)
week_df = df[mask].reset_index(drop=True)

if len(week_df) == 0:
    raise ValueError(f"No data found for week starting {WEEK_START}.")

y_pred_week   = model.predict(week_df[feature_cols].values)
y_actual_week = week_df[target_col].values
y_naive_week  = week_df["target_lag_24h"].values
timestamps    = week_df["datetime_utc12"].values

# ── SCHEDULE CHEAPEST HOURS PER DAY ──────────────────────────────────────────

schedule = []

for day_offset in range(7):
    day_start = week_start_ts + pd.Timedelta(days=day_offset)
    day_end   = day_start + pd.Timedelta(hours=24)

    day_mask = (week_df["datetime_utc12"] >= day_start) & (week_df["datetime_utc12"] < day_end)
    day_idx  = np.where(day_mask)[0]

    if len(day_idx) < 24:
        continue

    pred_day   = y_pred_week[day_idx]
    actual_day = y_actual_week[day_idx]
    naive_day  = y_naive_week[day_idx]

    model_chosen   = np.argsort(pred_day)[:HOURS_PER_DAY]
    naive_chosen   = np.argsort(naive_day)[:HOURS_PER_DAY]
    uniform_chosen = np.linspace(0, 23, HOURS_PER_DAY, dtype=int)

    model_cost   = actual_day[model_chosen].mean()   * MACHINE_POWER_MW * HOURS_PER_DAY
    naive_cost   = actual_day[naive_chosen].mean()   * MACHINE_POWER_MW * HOURS_PER_DAY
    uniform_cost = actual_day[uniform_chosen].mean() * MACHINE_POWER_MW * HOURS_PER_DAY

    schedule.append({
        "date":           day_start.date(),
        "pred_day":       pred_day,
        "actual_day":     actual_day,
        "naive_day":      naive_day,
        "model_chosen":   model_chosen,
        "naive_chosen":   naive_chosen,
        "uniform_chosen": uniform_chosen,
        "model_cost":     model_cost,
        "naive_cost":     naive_cost,
        "uniform_cost":   uniform_cost,
    })

# ── WEEKLY COST SUMMARY ───────────────────────────────────────────────────────

total_model   = sum(d["model_cost"]   for d in schedule)
total_naive   = sum(d["naive_cost"]   for d in schedule)
total_uniform = sum(d["uniform_cost"] for d in schedule)

print(f"\n{'─'*52}")
print(f"{'Date':<14} {'Model (NZD)':>12} {'Naive (NZD)':>12} {'Uniform (NZD)':>14}")
print(f"{'─'*52}")
for d in schedule:
    print(f"{str(d['date']):<14} {d['model_cost']:>12.2f} {d['naive_cost']:>12.2f} {d['uniform_cost']:>14.2f}")
print(f"{'─'*52}")
print(f"{'TOTAL':<14} {total_model:>12.2f} {total_naive:>12.2f} {total_uniform:>14.2f}")
print(f"\nSavings vs uniform — Model: {total_uniform - total_model:.2f} NZD  "
      f"({100*(total_uniform-total_model)/total_uniform:.1f}%)")
print(f"Savings vs uniform — Naive: {total_uniform - total_naive:.2f} NZD  "
      f"({100*(total_uniform-total_naive)/total_uniform:.1f}%)")
print(f"Model vs Naive     — Model saves: {total_naive - total_model:.2f} NZD  "
      f"({100*(total_naive-total_model)/total_naive:.1f}%)")

# ── VISUALISATION ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 7, figsize=(28, 8), sharey=False)
fig.suptitle(
    f"Optimal Charging — Week of {WEEK_START}  |  "
    f"{DAILY_ENERGY_MWH} MWh/day @ {MACHINE_POWER_MW} MW ({HOURS_PER_DAY}h/day)\n"
    f"Weekly cost — Model: {total_model:.0f} NZD  |  "
    f"Naive: {total_naive:.0f} NZD  |  Uniform: {total_uniform:.0f} NZD",
    fontsize=11
)

hours = np.arange(24)

strategies = [
    ("model",  "Model",  "darkorange",   "Model prediction", "darkorange"),
    ("naive",  "Naive",  "mediumpurple", "Naive (lag 24h)",  "mediumpurple"),
]

for row, (key, label, bar_color, line_label, line_color) in enumerate(strategies):
    for col, d in enumerate(schedule):
        ax = axes[row][col]

        chosen     = d[f"{key}_chosen"]
        cost       = d[f"{key}_cost"]
        price_line = d["pred_day"] if key == "model" else d["naive_day"]

        ax.bar(hours, d["actual_day"], color="lightsteelblue", alpha=0.5, label="Actual price")
        ax.bar(hours[chosen], d["actual_day"][chosen], color=bar_color, alpha=0.85,
               label=f"{label} picks")
        ax.step(hours, price_line, where="mid", color=line_color,
                linewidth=1.2, linestyle="--", label=line_label)

        day_str  = str(d["date"])
        cost_str = f"{cost:.1f} NZD"
        ax.set_title(f"{day_str}\ncost: {cost_str}", fontsize=8)
        ax.set_xlabel("Hour", fontsize=7)
        if col == 0:
            ax.set_ylabel(f"{label} strategy\nPrice (NZD/MWh)", fontsize=7)
        ax.tick_params(labelsize=6)
        if col == 0 and row == 0:
            ax.legend(fontsize=6, loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CHARGING SCHEDULER — ALL 52 WEEKS OF 2024
# Uses same DAILY_ENERGY_MWH and MACHINE_POWER_MW from the cell above.
# Positive saving = model/naive paid less than uniform (good).
# Negative saving = model/naive paid more than uniform (bad).
# ══════════════════════════════════════════════════════════════════════════════

weekly_savings = []

for week_start in pd.date_range("2024-01-01", "2024-12-25", freq="7D"):
    ws_ts  = week_start
    we_ts  = ws_ts + pd.Timedelta(days=7)

    mask    = (df["datetime_utc12"] >= ws_ts) & (df["datetime_utc12"] < we_ts)
    wdf     = df[mask].reset_index(drop=True)

    if len(wdf) < 168:
        continue

    y_pred_w   = model.predict(wdf[feature_cols].values)
    y_actual_w = wdf[target_col].values
    y_naive_w  = wdf["target_lag_24h"].values

    w_model   = 0.0
    w_naive   = 0.0
    w_uniform = 0.0

    for day_offset in range(7):
        ds = ws_ts + pd.Timedelta(days=day_offset)
        de = ds + pd.Timedelta(hours=24)

        dmask = (wdf["datetime_utc12"] >= ds) & (wdf["datetime_utc12"] < de)
        didx  = np.where(dmask)[0]

        if len(didx) < 24:
            continue

        pred_d   = y_pred_w[didx]
        actual_d = y_actual_w[didx]
        naive_d  = y_naive_w[didx]

        mc = np.argsort(pred_d)[:HOURS_PER_DAY]
        nc = np.argsort(naive_d)[:HOURS_PER_DAY]
        uc = np.linspace(0, 23, HOURS_PER_DAY, dtype=int)

        w_model   += actual_d[mc].mean() * MACHINE_POWER_MW * HOURS_PER_DAY
        w_naive   += actual_d[nc].mean() * MACHINE_POWER_MW * HOURS_PER_DAY
        w_uniform += actual_d[uc].mean() * MACHINE_POWER_MW * HOURS_PER_DAY

    weekly_savings.append({
        "week_start":      week_start.date(),
        "cost_model":      w_model,
        "cost_naive":      w_naive,
        "cost_uniform":    w_uniform,
        "saving_model":    w_uniform - w_model,   # + = cheaper than uniform
        "saving_naive":    w_uniform - w_naive,
        "model_vs_naive":  w_naive   - w_model,   # + = model cheaper than naive
    })

sdf = pd.DataFrame(weekly_savings)

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"Settings: {DAILY_ENERGY_MWH} MWh/day  @  {MACHINE_POWER_MW} MW  ({HOURS_PER_DAY}h/day)\n")
print(f"{'Week':>12}  {'Uniform':>10}  {'Model':>10}  {'Naive':>10}  "
      f"{'Save Model':>11}  {'Save Naive':>11}  {'Mdl vs Naive':>13}")
print("─" * 82)
for _, r in sdf.iterrows():
    print(f"{str(r['week_start']):>12}  {r['cost_uniform']:>10.2f}  {r['cost_model']:>10.2f}  "
          f"{r['cost_naive']:>10.2f}  {r['saving_model']:>+11.2f}  "
          f"{r['saving_naive']:>+11.2f}  {r['model_vs_naive']:>+13.2f}")
print("─" * 82)
print(f"{'TOTAL':>12}  {sdf['cost_uniform'].sum():>10.2f}  {sdf['cost_model'].sum():>10.2f}  "
      f"{sdf['cost_naive'].sum():>10.2f}  {sdf['saving_model'].sum():>+11.2f}  "
      f"{sdf['saving_naive'].sum():>+11.2f}  {sdf['model_vs_naive'].sum():>+13.2f}")

pct_m = 100 * sdf["saving_model"].sum()   / sdf["cost_uniform"].sum()
pct_n = 100 * sdf["saving_naive"].sum()   / sdf["cost_uniform"].sum()
pct_v = 100 * sdf["model_vs_naive"].sum() / sdf["cost_naive"].sum()
print(f"\nFull-year savings vs uniform — Model: {pct_m:+.1f}%   Naive: {pct_n:+.1f}%")
print(f"Full-year model vs naive               Model: {pct_v:+.1f}%")

# ── Figure ────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 1, figsize=(18, 8))
fig.suptitle(
    f"Weekly Charging Savings vs Uniform Spread — 2024\n"
    f"{DAILY_ENERGY_MWH} MWh/day @ {MACHINE_POWER_MW} MW ({HOURS_PER_DAY}h/day)  |  "
    f"Positive = cheaper than uniform",
    fontsize=12
)

weeks  = np.arange(len(sdf))
labels = [str(w) for w in sdf["week_start"]]

# ── Panel 1: Model vs Naive vs Uniform ────────────────────────────────────────
ax1 = axes[0]
w = 0.35
ax1.bar(weeks - w/2, sdf["saving_model"], width=w, color="darkorange",
        alpha=0.8, label="Model saving vs uniform")
ax1.bar(weeks + w/2, sdf["saving_naive"], width=w, color="mediumpurple",
        alpha=0.8, label="Naive saving vs uniform")
ax1.axhline(0, color="black", linewidth=0.8)
ax1.set_xticks(weeks[::2])
ax1.set_xticklabels(labels[::2], rotation=45, ha="right", fontsize=7)
ax1.set_ylabel("Saving (NZD)")
ax1.set_title("Weekly saving vs uniform spread (Model and Naive)")
ax1.legend()

# Running cumulative line
ax1b = ax1.twinx()
ax1b.plot(weeks, sdf["saving_model"].cumsum(), color="darkorange",
          linewidth=2, linestyle="-", label="Cumulative model saving")
ax1b.plot(weeks, sdf["saving_naive"].cumsum(), color="mediumpurple",
          linewidth=2, linestyle="--", label="Cumulative naive saving")
ax1b.set_ylabel("Cumulative saving (NZD)")
ax1b.legend(loc="upper left", fontsize=8)

# ── Panel 2: Model vs Naive (head-to-head) ────────────────────────────────────
ax2 = axes[1]
colors = ["darkorange" if v >= 0 else "steelblue" for v in sdf["model_vs_naive"]]
ax2.bar(weeks, sdf["model_vs_naive"], color=colors, alpha=0.8)
ax2.axhline(0, color="black", linewidth=0.8)
ax2.set_xticks(weeks[::2])
ax2.set_xticklabels(labels[::2], rotation=45, ha="right", fontsize=7)
ax2.set_ylabel("Saving (NZD)")
ax2.set_title("Model vs Naive head-to-head  (orange = model wins, blue = naive wins)")

ax2b = ax2.twinx()
ax2b.plot(weeks, sdf["model_vs_naive"].cumsum(), color="black",
          linewidth=2, label="Cumulative model advantage")
ax2b.set_ylabel("Cumulative (NZD)")
ax2b.legend(loc="upper left", fontsize=8)

plt.tight_layout()
plt.show()